# RNA-seq Statistical Modeling — Hypothesis Testing, GLMs, and Multiple Testing

This notebook builds a working understanding of the statistics behind `DESeq2`, using **real output** from the [rnaseq-snakemake](../../pipelines/rnaseq-snakemake/) pipeline — real yeast RNA-seq data (GSE110004), comparing wild-type (`WT`) against Rap1 transcription factor depletion (`RAP1_IAA`), 3 biological replicates per group.

Three real files feed this notebook:
- `results.csv` — DESeq2's per-gene summary statistics (baseMean, log2FoldChange, pvalue, padj)
- `counts_normalized.csv` — per-sample normalized read counts, one row per gene
- `samples.tsv` — the sample sheet (sample → condition mapping)

Goal: understand *what DESeq2 is actually computing* well enough to explain it, not just read its output.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Real pipeline output, not synthetic — three files produced by the
# rnaseq-snakemake run on real yeast data (WT vs RAP1_IAA).
PIPELINE = "../../pipelines/rnaseq-snakemake/results/deseq2"

# DESeq2's per-gene summary: baseMean, log2FoldChange, pvalue, padj.
# index_col=0 makes the gene name the row index (not a plain column),
# so later code can do results.loc["YAR009C", "padj"] directly.
results = pd.read_csv(f"{PIPELINE}/results.csv", index_col=0)

# Per-sample normalized counts — one row per gene, one column per sample.
# This is the raw material Section 1/2's own hypothesis tests run on;
# results.csv alone only has DESeq2's already-computed summary stats.
counts = pd.read_csv(f"{PIPELINE}/counts_normalized.csv", index_col=0)

# The sample sheet: which sample belongs to which condition (WT/RAP1_IAA).
samples = pd.read_csv("../../pipelines/rnaseq-snakemake/config/samples.tsv", sep="\t")

# Cosmetic: name the index so it prints as "gene" instead of "Unnamed: 0".
results.index.name = "gene"
counts.index.name = "gene"

print(f"{results.shape[0]} genes tested, {counts.shape[1]} samples")
results.sort_values("padj").head()

In [17]:
print(results.head(5))

          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
gene                                                                      
HRA1      9.343098        0.123319  0.687355  0.179412  0.857615  0.998505
snR18     5.573197        0.902860  0.935128  0.965494  0.334297  0.998505
tA(UGC)A  0.000000             NaN       NaN       NaN       NaN       NaN
tL(CAA)A  0.000000             NaN       NaN       NaN       NaN       NaN
tP(UGG)A  0.201268        1.460263  4.080473  0.357866  0.720443  0.998505


In [9]:
# print(counts.head(5))

In [10]:
# print(samples.head(5))

## 1. Hypothesis Testing Refresher

For every gene, DESeq2 is answering the same question: **is this gene's expression level in `RAP1_IAA` samples different from `WT` samples, more than we'd expect from random noise between replicates alone?**

Formally, for each gene:
- **Null hypothesis (H₀):** no real difference in mean expression between conditions — any observed difference is just sampling noise.
- **Alternative hypothesis (H₁):** there is a real difference.

Two classic tests can answer this kind of question for two independent groups:

- **Independent two-sample t-test** — assumes the data in each group is roughly normally distributed. Compares the difference in group means, scaled by pooled variance.
- **Mann-Whitney U / Wilcoxon rank-sum test** — a *non-parametric* alternative: makes no assumption about the underlying distribution, works directly on ranks instead of raw values.

**Why this matters for RNA-seq specifically:** with only 3 replicates per group, there isn't nearly enough data to actually verify a normality assumption — and raw read counts are discrete, non-negative, and often right-skewed (a few highly expressed genes, many lowly expressed ones), which is a poor match for the normal distribution's assumptions to begin with. This is exactly why DESeq2 doesn't use a plain t-test — it models counts with a **negative binomial** distribution instead (Section 2). But it's worth seeing what a plain t-test and a rank-based test *would* say first, as a baseline to compare against.

In [ ]:
# Split the 6 sample names into the two condition groups, using the sample
# sheet as the source of truth (same "condition" column DESeq2's own
# design formula ~condition uses).
condition_map = samples.set_index("sample")["condition"]
wt_samples = condition_map[condition_map == "WT"].index.tolist()
treated_samples = condition_map[condition_map == "RAP1_IAA"].index.tolist()

def compare_gene(gene):
    # Pull just this one gene's row, split into its two group's raw values —
    # 3 numbers per group, exactly what a real small RNA-seq experiment gives you.
    wt_vals = counts.loc[gene, wt_samples].values
    treated_vals = counts.loc[gene, treated_samples].values

    # Two-sample t-test: assumes normally-distributed data in each group.
    t_stat, t_p = stats.ttest_ind(wt_vals, treated_vals)

    # Mann-Whitney U / Wilcoxon rank-sum: no distribution assumption,
    # works on ranks instead of raw values.
    u_stat, u_p = stats.mannwhitneyu(wt_vals, treated_vals, alternative="two-sided")

    # DESeq2's own numbers for the same gene, for direct comparison.
    return pd.Series({
        "WT_mean": wt_vals.mean(),
        "RAP1_IAA_mean": treated_vals.mean(),
        "t_test_p": t_p,
        "wilcoxon_p": u_p,
        "DESeq2_p": results.loc[gene, "pvalue"],
        "DESeq2_padj": results.loc[gene, "padj"],
    })

# One gene DESeq2 called highly significant, one it called not significant at all —
# run compare_gene() on each and stack the results into one table.
example_genes = ["YAR009C", "YAL005C"]
comparison = pd.DataFrame({g: compare_gene(g) for g in example_genes}).T
comparison

**Look at the table above.** All three methods (t-test, Wilcoxon, DESeq2) should broadly agree on *which* gene looks different between conditions and which doesn't — that agreement itself is a useful sanity check. But also notice: with only n=3 per group, a plain Wilcoxon rank-sum test has very limited resolution (there are only $\binom{6}{3}=20$ possible ways to split 6 values into two groups of 3, so its smallest possible p-value is mechanically bounded — it literally cannot report a p-value as small as DESeq2's). This is a genuine, practical reason RNA-seq analysis doesn't just run one test per gene with `scipy` and call it done — it needs a model built specifically for small-sample, discrete count data. That model is the negative binomial GLM, covered next.

## 2. The Negative Binomial GLM — What DESeq2 Actually Fits

RNA-seq read counts are discrete, non-negative integers — a poor fit for a normal-distribution t-test's assumptions. The natural first choice for count data is the **Poisson distribution** — but Poisson has a restrictive property: variance always equals the mean, exactly, by definition. Real biological replicates practically always show **more variance than that** (*overdispersion*) — no two yeast cultures behave identically even under the same condition, adding real biological variability on top of pure counting noise.

The **negative binomial distribution** generalizes Poisson with one extra parameter, dispersion ($\alpha$), that explicitly models this extra variance:

$$\text{Var}(\text{count}) = \mu + \alpha \mu^2$$

where $\mu$ is the mean count. When $\alpha = 0$, this collapses back to Poisson exactly. DESeq2 fits, for every gene, a **generalized linear model (GLM)**:

$$\log(\mu) = \beta_0 + \beta_1 \cdot \text{condition}$$

a log link connecting the linear predictor to the mean count, where $\beta_1$ — the coefficient on `condition` — is directly the **log fold change** already seen in `results.csv` (natural log; DESeq2 reports it converted to log2).

Let's fit this ourselves, from scratch, for `YAR009C` — using only that one gene's 6 real count values, with no cross-gene borrowing this time — and compare the result against both Section 1's naive tests and DESeq2's actual (shrinkage-informed) fit.

In [ ]:
import statsmodels.api as sm

def fit_nb_glm(gene):
    df = samples.copy()
    # Attach this one gene's 6 count values onto the sample sheet, so each
    # row is (sample, condition, count) — the shape a regression needs.
    df["count"] = counts.loc[gene, df["sample"]].values

    # DESeq2 (via R's factor()) orders levels alphabetically, making RAP1_IAA
    # the reference level here ("R" < "W") — match that explicitly so our
    # coefficient's sign lines up with DESeq2's log2FoldChange directly.
    df["condition"] = pd.Categorical(df["condition"], categories=["RAP1_IAA", "WT"])

    # Turn the condition column into a 0/1 indicator: drop_first=True drops
    # the reference level (RAP1_IAA), leaving one column named "WT"
    # (1 = WT sample, 0 = RAP1_IAA sample) — this becomes beta_1 in the model.
    X = pd.get_dummies(df["condition"], drop_first=True).astype(float)
    X = sm.add_constant(X)  # adds the intercept column (beta_0)
    y = df["count"]

    # Fit log(count) = beta_0 + beta_1*WT as a negative binomial GLM.
    # disp=0 just silences the optimizer's convergence printout.
    fit = sm.NegativeBinomial(y, X).fit(disp=0)

    return pd.Series({
        "baseMean": results.loc[gene, "baseMean"],
        # fit.params["WT"] is beta_1 in natural-log units; DESeq2 reports
        # log2, so divide by ln(2) to convert onto the same scale.
        "our_NB_log2FC": fit.params["WT"] / np.log(2),
        "our_NB_pvalue": fit.pvalues["WT"],
        # statsmodels' NegativeBinomial estimates alpha (dispersion) jointly
        # with the coefficients via maximum likelihood — no cross-gene
        # information used, unlike DESeq2's shrinkage-based estimate.
        "our_NB_alpha": fit.params["alpha"],
        "DESeq2_log2FC": results.loc[gene, "log2FoldChange"],
        "DESeq2_pvalue": results.loc[gene, "pvalue"],
    })

# Deliberately spread across baseMean, not just DESeq2's top hit: a strong
# high-count gene, a moderate significant one, and a borderline low-count
# one — so the shrinkage comparison isn't resting on a single cherry-picked
# example.
example_genes = ["YAR009C", "YAL003W", "YAL061W"]
glm_comparison = pd.DataFrame({g: fit_nb_glm(g) for g in example_genes}).T
glm_comparison

**What actually happens here, from the real numbers** (worth noting: the opposite of a natural first guess, corrected below): for `YAL003W` and `YAL061W`, our from-scratch fit is *dramatically more* "significant" than DESeq2 — not weaker. Only `YAR009C` (both p-values astronomically tiny either way) is close.

**Why:** maximum-likelihood estimates of a dispersion parameter from very few data points (n=3 per group) are systematically **biased toward underestimating** true variability — the same general phenomenon behind sample variance dividing by n−1 instead of n. An underestimated dispersion makes a model think the data is cleaner than it really is, producing p-values that are too small — the naive fit is *overconfident*, not just noisy.

DESeq2's empirical Bayes shrinkage exists specifically to correct this: it pulls each gene's likely-too-low naive dispersion estimate up toward the more reliable, dataset-wide trend, making DESeq2 appropriately **more conservative** than a naive per-gene fit — not more powerful, which is the easy but wrong intuition. This is the actual historical motivation for shrinkage-based methods in genomics (DESeq2, edgeR, limma's eBayes): early per-gene-only analyses were notorious for producing floods of false positives, precisely because small-sample dispersion estimates run overconfident by default. Shrinkage exists to prevent that, not to manufacture extra significance.

Compare `our_NB_alpha` against how confident each p-value is: the smaller our naive alpha, the more overconfident our test — and DESeq2's shrinkage correction shows up largest exactly where the naive estimate is furthest from realistic.

## 3. Multiple Testing Correction — Why RNA-seq Can't Use Raw p-values

Every gene tested is one independent statistical test. With 124 genes tested at the conventional threshold α = 0.05, **pure chance alone** predicts about 124 × 0.05 ≈ 6 genes would look "significant" even if RAP1 depletion had *zero* real effect anywhere — 5% of independent tests cross the threshold by luck alone, by definition. Real experiments often test thousands to tens of thousands of genes, where that "chance alone" count climbs into the hundreds. Raw p-values, used naively across many simultaneous tests, systematically overstate how many real findings you actually have.

In [ ]:
# Only count genes that actually got tested — a few genes had zero reads in
# every sample (see the tA(UGC)A / tL(CAA)A NaN rows earlier), and DESeq2
# leaves their pvalue as NaN rather than testing them.
n_genes = results["pvalue"].notna().sum()

# If every gene's true effect were exactly zero, we'd still expect 5% of
# n_genes tests to cross p < 0.05 purely from random sampling noise —
# that's the literal definition of a 0.05 significance threshold.
expected_by_chance = n_genes * 0.05

raw_sig = (results["pvalue"] < 0.05).sum()          # naive, uncorrected count
adj_sig = (results["padj"] < 0.05).sum()             # DESeq2's BH-corrected count

pd.Series({
    "genes tested": n_genes,
    "expected 'significant' by chance alone (n x 0.05)": expected_by_chance,
    "genes with raw pvalue < 0.05": raw_sig,
    "genes with BH-adjusted padj < 0.05": adj_sig,
})

Two classic corrections address this, with very different philosophies:

- **Bonferroni correction** — controls the *family-wise error rate* (FWER): the probability of making **even one** false positive across all tests. Simple but blunt: divide α by the number of tests (α/n), equivalent to multiplying every p-value by n. Extremely conservative once n is in the thousands — real experiments testing 20,000+ genes would need a p-value smaller than 0.05/20000 = 0.0000025 just to pass, strict enough to miss most true findings.

- **Benjamini-Hochberg (BH) / False Discovery Rate (FDR)** — a fundamentally different question: instead of "what's the chance of *any* false positive across everything," it controls "of the genes I call significant, what *fraction* are expected to be false positives?" Setting `padj < 0.05` means: among everything called significant, on average 5% might be false positives — a controlled, known error *rate*, not zero errors. This tradeoff is what makes it practical for genomics: accept a small, quantified false-positive rate in exchange for actually being able to detect real effects at all.

`padj` in `results.csv` is DESeq2's BH-corrected value. Let's implement BH from scratch and confirm it reproduces DESeq2's own numbers.

In [ ]:
def benjamini_hochberg(pvalues):
    pvalues = pvalues.dropna()          # skip untested genes (NaN pvalue)
    n = len(pvalues)

    # Step 1: sort p-values smallest to largest — BH is defined in terms
    # of each p-value's *rank* among all the others, not its raw value alone.
    ranked = pvalues.sort_values()
    ranks = np.arange(1, n + 1)         # 1, 2, 3, ..., n — rank of each sorted p-value

    # Step 2: the core BH formula — scale each p-value by (total tests / its rank).
    # The smallest p-value (rank 1) gets scaled up the most; the largest
    # (rank n) is left roughly as-is. This is what makes BH less strict
    # than Bonferroni (which scales every p-value by n, not n/rank).
    bh = ranked.values * n / ranks

    # Step 3: enforce monotonicity. Raw BH values aren't guaranteed to be
    # non-decreasing as you move to larger raw p-values, which wouldn't
    # make sense for an "adjusted p-value" — so walk from the largest
    # p-value down to the smallest, taking a running minimum. Reversing
    # twice ([::-1]) applies the running-min back-to-front, then flips
    # the result back to the original (ascending) order.
    bh = np.minimum.accumulate(bh[::-1])[::-1]

    bh = np.clip(bh, 0, 1)              # adjusted p-values can't exceed 1
    return pd.Series(bh, index=ranked.index)

our_padj = benjamini_hochberg(results["pvalue"])

# Compare our from-scratch values against DESeq2's own padj column,
# reindexed to the same gene order for a clean side-by-side.
check = pd.DataFrame({
    "our_BH_padj": our_padj,
    "DESeq2_padj": results.loc[our_padj.index, "padj"],
})
check["difference"] = (check["our_BH_padj"] - check["DESeq2_padj"]).abs()
check.sort_values("DESeq2_padj").head(10)

`difference` should be at (or extremely near) zero for every gene — unlike the GLM section, BH correction is a fully deterministic algorithm computed directly from the p-values, no model-fitting or estimation involved, so reproducing it exactly is expected here, not a coincidence.

One more comparison worth seeing directly — how much more conservative Bonferroni would be on this same data:

In [ ]:
# Bonferroni: divide alpha by the number of tests, rather than scaling by
# rank the way BH does — every gene gets the same strict cutoff regardless
# of how it ranks among the others.
bonferroni_sig = (results["pvalue"] < 0.05 / n_genes).sum()

pd.Series({
    "significant at raw p < 0.05": raw_sig,
    "significant at BH padj < 0.05": adj_sig,
    "significant at Bonferroni-corrected p < 0.05": bonferroni_sig,
})

On this dataset the gap between BH and Bonferroni is modest — only 124 genes tested, and the real hits here have unusually strong effects. Scale this up to a real experiment testing 20,000+ human genes, and Bonferroni's strictness routinely eliminates genuine, moderate-effect genes that BH correctly keeps — the gap widens dramatically with test count, not effect size. This is exactly why essentially every RNA-seq differential expression tool (DESeq2, edgeR, limma) reports BH/FDR-adjusted p-values by default, not Bonferroni-adjusted ones.